# Exercise: OT flow matching VISp → VISrl (Allen Ca)

This notebook is the **exercise** companion to `signal_walkthrough.ipynb`. Pipeline is the same (data → z-score → train → integrate → decode / MSE / PCA), except **you implement the flow-matching loss** yourself.

**Your task:** Fill in `cfm_loss_bridge` below. The rest of the notebook imports **`MLP_Residual`** and **`integrate_euler`** from `signal_model.py` (same as the solution walkthrough).

| Resource | Role |
|----------|------|
| This notebook | You implement **`cfm_loss_bridge`** |
| `signal_walkthrough.ipynb` | Full worked solution |
| `signal_model.py` | Reference implementation of the loss (after you try) |
| `synthetic_gaussian/solution/gaussian_model.py` | Same OT path for **noise → x1** (`cfm_loss`); your bridge replaces Gaussian **x0** with **VISp z**. |


## 1. Setup

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt


def _find_signal_flow_dir() -> Path:
    """Works from signal_flow/, Flow_Hackathon/, or repo root (e.g. flow/)."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        for rel in (
            Path("signal_model.py"),
            Path("signal_flow") / "signal_model.py",
            Path("Flow_Hackathon") / "signal_flow" / "signal_model.py",
        ):
            p = d / rel
            if p.is_file():
                return p.parent
    raise FileNotFoundError(
        "Could not find signal_model.py. cd to Flow_Hackathon/signal_flow or open the notebook from that folder."
    )


def _load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


FLOW_DIR = _find_signal_flow_dir()
if str(FLOW_DIR) not in sys.path:
    sys.path.insert(0, str(FLOW_DIR))

_sm = _load_module("signal_model", FLOW_DIR / "signal_model.py")
_sd = _load_module("signal_decode", FLOW_DIR / "signal_decode.py")

MLP_Residual = _sm.MLP_Residual
integrate_euler = _sm.integrate_euler
allen_frame_id_decode = _sd.allen_frame_id_decode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("signal_flow:", FLOW_DIR)



## 2. Load Ca embeddings (train / test)

Paths under `Flow_Hackathon/data/allen_data/` (paired trial order per seed).

In [ ]:
DATA_DIR = (FLOW_DIR.parent / "data" / "allen_data").resolve()
seed = 333

paths = {
    "visp_train": DATA_DIR / f"VISp_joint_trained_ca_{seed}_train.npy",
    "visp_test": DATA_DIR / f"VISp_joint_trained_ca_{seed}_test.npy",
    "visrl_train": DATA_DIR / f"VISrl_joint_trained_ca_{seed}_train.npy",
    "visrl_test": DATA_DIR / f"VISrl_joint_trained_ca_{seed}_test.npy",
}
for k, p in paths.items():
    assert p.is_file(), f"Missing {k}: {p}"

visp_train = np.load(paths["visp_train"]).astype(np.float32)
visp_test = np.load(paths["visp_test"]).astype(np.float32)
visrl_train = np.load(paths["visrl_train"]).astype(np.float32)
visrl_test = np.load(paths["visrl_test"]).astype(np.float32)

assert visp_train.shape[0] == visrl_train.shape[0]
assert visp_test.shape[0] == visrl_test.shape[0]
assert visp_train.shape[1] == visrl_train.shape[1]

for name, a in [
    ("VISp train", visp_train),
    ("VISp test", visp_test),
    ("VISrl train", visrl_train),
    ("VISrl test", visrl_test),
]:
    print(f"{name}: shape {a.shape}")



## 3. Z-normalize VISp and VISrl (each fit on its own train)

**VISp** \(z\) = flow start \(x_0\). **VISrl** \(z\) = target \(x_1\) and decode space.

In [ ]:
eps = 1e-6


def standardize(x, mu, std):
    return (x - mu) / std


mu_visp, std_visp = visp_train.mean(0), visp_train.std(0) + eps
mu_visrl, std_visrl = visrl_train.mean(0), visrl_train.std(0) + eps

z_visp_tr = standardize(visp_train, mu_visp, std_visp)
z_visp_te = standardize(visp_test, mu_visp, std_visp)
z_visrl_tr = standardize(visrl_train, mu_visrl, std_visrl)
z_visrl_te = standardize(visrl_test, mu_visrl, std_visrl)

z_visp_tr_t = torch.from_numpy(z_visp_tr).to(device)
z_visp_te_t = torch.from_numpy(z_visp_te).to(device)
z_visrl_tr_t = torch.from_numpy(z_visrl_tr).to(device)
z_visrl_te_t = torch.from_numpy(z_visrl_te).to(device)

dim = z_visp_tr.shape[1]
print("dim:", dim, "| paired train:", z_visp_tr_t.shape, z_visrl_tr_t.shape)



## Exercise — implement `cfm_loss_bridge`

Implement **optimal-transport conditional flow matching** for **paired** endpoints \(x_0 \to x_1\) (here: VISp \(z\) and VISrl \(z\) with the same row index). It is the same algebraic path as `synthetic_gaussian/solution/gaussian_model.py` **`cfm_loss`**, but **`x0` is not Gaussian noise** — it is a data batch.

**Given:** `model` is `MLP_Residual`; `x0`, `x1` are `[B, D]` tensors on the same device/dtype.

**Steps (hints):**

1. Define `SIGMA_MIN = 1e-4` (match the Gaussian solution notebook).
2. Sample `t \sim \mathrm{Uniform}(0,1)` with shape `(B,)`, same device/dtype as `x0`.
3. Let `sigma_t = 1 - (1 - SIGMA_MIN) * t` and broadcast so you can form the state  
   \(x_t = \sigma_t \odot x_0 + t \odot x_1\).
4. Target vector field (what the network should match):  
   \(v^* = x_1 - (1 - \texttt{SIGMA_MIN})\, x_0\).
5. Forward pass: `pred = model(x_t, t)` (watch `t` shape if the model expects 1D times).
6. Return **`((pred - v_star) ** 2).mean()`**.

**Do not** import `cfm_loss_bridge` from `signal_model` in this cell — implement it yourself. Afterward you may compare with `signal_model.cfm_loss_bridge` for debugging.


The CFM loss under this path becomes:

$$
\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t,\, x_1 \sim q(x_1),\, x_0 \sim p(x_0)} \left\| v_t(\psi_t(x_0);\theta) - \left(x_1 - (1-\sigma_{\min})x_0\right) \right\|^2
$$

with $\sigma_{\min} = 10^{-4}$ throughout and conditional flow $\psi_t(x_0) = (1 - (1-\sigma_{\min})t)\, x_0 + t x_1$

In [ ]:
SIGMA_MIN = 1e-4  # keep in sync with gaussian_model / signal_model


def cfm_loss_bridge(model, x0: torch.Tensor, x1: torch.Tensor) -> torch.Tensor:
    """
    OT conditional flow matching loss for paired (x0, x1), e.g. VISp z and VISrl z.

    Returns scalar MSE between model(x_t, t) and the target velocity.
    """
    # --- Your implementation here ---
    raise NotImplementedError("Replace this line with your OT-CFM loss (see markdown above).")



## 4. Train `MLP_Residual` (uses your loss)

Minibatch paired \((x_0, x_1)\) from **VISp** / **VISrl** train \(z\)-scores at the same indices; call **your** `cfm_loss_bridge`.


In [ ]:
model = MLP_Residual(
    input_size=dim,
    hidden_size=512,
    amount_layers=5,
    output_size=dim,
    time_dimension=128,
).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

BATCH = 512
STEPS = 3000
n_pair = z_visp_tr_t.shape[0]
assert n_pair == z_visrl_tr_t.shape[0]

losses = []
model.train()
for step in range(STEPS):
    idx = torch.randint(0, n_pair, (BATCH,), device=device)
    x0_b = z_visp_tr_t[idx]
    x1_b = z_visrl_tr_t[idx]
    loss = cfm_loss_bridge(model, x0_b, x1_b)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 500 == 0 or step == STEPS - 1:
        print(f"step {step:5d}  loss {loss.item():.6f}")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("MSE")
plt.title("OT-CFM bridge (VISp → VISrl train) — your loss")
plt.tight_layout()
plt.show()



## 5. Integrate VISp test → VISrl z-space

Euler `integrate_euler` from `signal_model.py` starting at **VISp test** \(z\)-scores.


In [ ]:
n_te = z_visp_te_t.shape[0]
assert n_te == z_visrl_te_t.shape[0]

with torch.no_grad():
    x_hat = integrate_euler(model, z_visp_te_t, n_steps=100)

z_flow_test = x_hat.cpu().numpy()
print("Flow output shape (VISrl z-space):", z_flow_test.shape)



## 6. Frame-ID decoding (`modality='ca'`)

Same protocol as `signal_walkthrough.ipynb`: **VISrl** train bank; oracle **`z_visrl_te`** vs flow **`z_flow_test`**.

### How kNN accuracy is computed (`signal_decode.allen_frame_id_decode`)

1. **Classifier:** `KNeighborsClassifier` with **cosine** distance.
2. **Choosing \(k\):** grid of `n_neighbors`; **8/9** of train rows fit the kNN, **1/9** used to pick \(k\) (lowest L1 frame error on that holdout).
3. **Test:** refit on **full train**, predict test.
4. **Accuracy %:** fraction of trials with \(|\hat{f} - f| < 30\) frames (coarse frame ID).

### 6b. MSE (later cell)

Flow endpoints vs **VISrl test** in \(z\)-space.


In [ ]:
train_labels = np.tile(np.arange(900), 9)
test_labels = np.arange(900)
assert len(train_labels) == z_visrl_tr.shape[0]
assert len(test_labels) == z_visrl_te.shape[0]

_, _, acc_oracle = allen_frame_id_decode(
    z_visrl_tr,
    train_labels,
    z_visrl_te,
    test_labels,
    modality="ca",
    decoder="knn",
)

_, _, acc_flow = allen_frame_id_decode(
    z_visrl_tr,
    train_labels,
    z_flow_test,
    test_labels,
    modality="ca",
    decoder="knn",
)

print(f"Oracle  VISrl train → VISrl test:             {acc_oracle:.2f}%")
print(f"Flow    VISp test→Euler→VISrl bank decode:   {acc_flow:.2f}%")



In [ ]:
# MSE in VISrl z-space: flow endpoint vs held-out VISrl test (same trial pairing)
assert z_flow_test.shape == z_visrl_te.shape
diff = z_flow_test - z_visrl_te
mse = float(np.mean(diff ** 2))
rmse_elem = float(np.sqrt(mse))
per_trial_mse = np.mean(diff ** 2, axis=1)

print(f"MSE (mean over trials × dims):     {mse:.6f}")
print(f"RMSE (√MSE over all entries):        {rmse_elem:.6f}")
print(f"per-trial MSE — mean: {float(per_trial_mse.mean()):.6f},  median: {float(np.median(per_trial_mse)):.6f}")



## 7. Optional: 2D PCA glimpse

**VISrl train** (subsample), **VISrl test**, **flow** endpoints in one plane.


In [ ]:
from sklearn.decomposition import PCA

Z = np.vstack([z_visrl_tr[::80], z_visrl_te, z_flow_test])
pca = PCA(n_components=2, random_state=0).fit(Z)


def proj(A):
    return pca.transform(A)


plt.figure(figsize=(7, 6))
plt.scatter(*proj(z_visrl_tr[::80]).T, s=8, alpha=0.6, label="VISrl train (subsample)")
plt.scatter(*proj(z_visrl_te).T, s=8, alpha=0.6, label="VISrl test (oracle)")
plt.scatter(*proj(z_flow_test).T, s=8, alpha=0.6, label="Flow (VISp test → Euler)")
plt.legend()
plt.title("PCA: VISrl + flow (VISp→VISrl)")
plt.axis("off")
plt.tight_layout()
plt.show()

